# Email Spam Detection — KNN vs SVM

Binary classification: **0 = Not Spam**, **1 = Spam**.

In [ ]:
!pip -q install pandas numpy matplotlib seaborn scikit-learn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import files

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score, roc_curve
)

import warnings
warnings.filterwarnings('ignore')


In [ ]:
# Upload the Kaggle CSV file
uploaded = files.upload()


In [ ]:
# Load dataset
file_name = next(iter(uploaded))
df = pd.read_csv(file_name)

print("Dataset shape:", df.shape)
display(df.head())


## 1. Data Preprocessing

In [ ]:
# Remove completely empty columns and duplicate rows
df = df.dropna(axis=1, how='all')
df = df.drop_duplicates().reset_index(drop=True)

id_col = df.columns[0]
target_col = df.columns[-1]

print("ID column:", id_col)
print("Target column:", target_col)

# First column is an email identifier; last column is the label.
X = df.iloc[:, 1:-1].copy()
y = pd.to_numeric(df.iloc[:, -1], errors='coerce')

# Keep only valid binary labels
valid = y.isin([0, 1])
X = X.loc[valid].copy()
y = y.loc[valid].astype(int)

# Convert word counts to numeric and handle missing values
X = X.apply(pd.to_numeric, errors='coerce').fillna(0)

print("Feature matrix:", X.shape)
print("Target:", y.shape)
print("\nMissing values:", X.isnull().sum().sum())


In [ ]:
# Class distribution
print(y.value_counts())
print("\nClass percentages:")
print((y.value_counts(normalize=True) * 100).round(2))

plt.figure(figsize=(6,4))
sns.countplot(x=y)
plt.xticks([0,1], ["Not Spam", "Spam"])
plt.title("Email Class Distribution")
plt.show()


## 2. Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", X_train.shape)
print("Testing samples :", X_test.shape)


## 3. Feature Scaling

In [ ]:
# Scaling is important for distance-based KNN and SVM.
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Feature scaling completed.")


## 4. K-Nearest Neighbors (KNN)

In [ ]:
knn = KNeighborsClassifier(
    n_neighbors=5,
    weights='distance',
    n_jobs=-1
)

knn.fit(X_train_scaled, y_train)

knn_pred = knn.predict(X_test_scaled)
knn_prob = knn.predict_proba(X_test_scaled)[:, 1]

print(classification_report(
    y_test,
    knn_pred,
    target_names=["Not Spam", "Spam"]
))


## 5. Support Vector Machine (SVM)

In [ ]:
svm = SVC(
    kernel='rbf',
    C=1.0,
    gamma='scale',
    probability=True,
    random_state=42
)

svm.fit(X_train_scaled, y_train)

svm_pred = svm.predict(X_test_scaled)
svm_prob = svm.predict_proba(X_test_scaled)[:, 1]

print(classification_report(
    y_test,
    svm_pred,
    target_names=["Not Spam", "Spam"]
))


## 6. Model Performance Comparison

In [ ]:
def metrics(y_true, y_pred, y_prob):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1 Score": f1_score(y_true, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_true, y_prob)
    }

results = pd.DataFrame({
    "KNN": metrics(y_test, knn_pred, knn_prob),
    "SVM": metrics(y_test, svm_pred, svm_prob)
}).T

display(results.round(4))


## 7. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12,5))

cm_knn = confusion_matrix(y_test, knn_pred)
cm_svm = confusion_matrix(y_test, svm_pred)

sns.heatmap(
    cm_knn, annot=True, fmt='d', cmap='Blues',
    xticklabels=["Not Spam", "Spam"],
    yticklabels=["Not Spam", "Spam"],
    ax=axes[0]
)
axes[0].set_title("KNN Confusion Matrix")
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("Actual")

sns.heatmap(
    cm_svm, annot=True, fmt='d', cmap='Greens',
    xticklabels=["Not Spam", "Spam"],
    yticklabels=["Not Spam", "Spam"],
    ax=axes[1]
)
axes[1].set_title("SVM Confusion Matrix")
axes[1].set_xlabel("Predicted")
axes[1].set_ylabel("Actual")

plt.tight_layout()
plt.show()


## 8. ROC Curve

In [ ]:
knn_fpr, knn_tpr, _ = roc_curve(y_test, knn_prob)
svm_fpr, svm_tpr, _ = roc_curve(y_test, svm_prob)

plt.figure(figsize=(8,6))
plt.plot(
    knn_fpr, knn_tpr,
    label=f"KNN (AUC = {roc_auc_score(y_test, knn_prob):.3f})"
)
plt.plot(
    svm_fpr, svm_tpr,
    label=f"SVM (AUC = {roc_auc_score(y_test, svm_prob):.3f})"
)
plt.plot([0,1], [0,1], linestyle='--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve — Spam Detection")
plt.legend()
plt.show()


## 9. Metric Comparison

In [ ]:
results.plot(kind='bar', figsize=(10,6))
plt.ylim(0, 1.05)
plt.ylabel("Score")
plt.title("KNN vs SVM Performance")
plt.xticks(rotation=0)
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()


## 10. Predict a New Email

In [ ]:
# Demonstration using one test email.
# A real new email must first be converted to the same 3000 word-count features.

sample_email = X_test.iloc[[0]]
sample_scaled = scaler.transform(sample_email)

knn_result = knn.predict(sample_scaled)[0]
svm_result = svm.predict(sample_scaled)[0]

print("Actual label :", "Spam" if y_test.iloc[0] == 1 else "Not Spam")
print("KNN prediction:", "Spam" if knn_result == 1 else "Not Spam")
print("SVM prediction:", "Spam" if svm_result == 1 else "Not Spam")


## Conclusion

The notebook evaluates KNN and SVM using Accuracy, Precision, Recall, F1 Score, ROC-AUC, and confusion matrices.